# Azure AI Evaluations SDK - Comprehensive Tutorial

Welcome to the Azure AI Evaluations SDK tutorial! This notebook will guide you through everything you need to know to evaluate your AI applications, from basic setup to advanced evaluation workflows.

## What You'll Learn

1. **Setup and Installation** - Install SDK and configure environment
2. **Authentication** - Connect to Azure AI Foundry services
3. **Built-in Evaluators** - Use pre-built metrics for quality and safety
4. **Custom Evaluators** - Create your own evaluation functions
5. **Batch Evaluation** - Run evaluations on datasets
6. **Results Analysis** - Interpret and visualize evaluation results
7. **Integration** - Connect with Azure OpenAI and PromptFlow

## Prerequisites

- Python 3.8 or later
- Azure subscription with Azure AI Foundry access
- Basic Python programming knowledge

Let's get started!

## 1. Setup and Installation

First, we'll install the Azure AI Evaluation SDK and its dependencies. The SDK provides several installation options:

- **Base SDK**: Core evaluation functionality
- **Remote extras**: Cloud-based evaluations in Azure AI Foundry
- **Redteam extras**: Adversarial testing capabilities

In [ ]:
# Install the Azure AI Evaluation SDK with remote capabilities
%pip install azure-ai-evaluation[remote]

# Install additional required packages
%pip install azure-ai-projects azure-identity promptflow-azure python-dotenv pandas

### Import Required Libraries

Now let's import all the libraries we'll use throughout this tutorial.

In [ ]:
import os
import json
from dotenv import load_dotenv
import pandas as pd

%load_ext autoreload
%autoreload 2


load_dotenv()  # Load environment variables from .env file

from rich import console
console = console.Console()

from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_CLIENT_ID = os.getenv("AZURE_CLIENT_ID")

# Example usage:
if False:
    print(f"Project Endpoint: {AZURE_AI_PROJECT_ENDPOINT}")
    print(f"Model Deployment Name: {MODEL_DEPLOYMENT_NAME}")
    print(f"OpenAI Endpoint: {AZURE_OPENAI_ENDPOINT}")
    print(f"OpenAI API Key: {AZURE_OPENAI_API_KEY}")
    print(f"OpenAI API Version: {AZURE_OPENAI_API_VERSION}")
    print(f"AI Resource Name: {AZURE_AI_RESOURCE_NAME}")
    print(f"Client ID: {AZURE_CLIENT_ID}")



# Azure Identity for authentication
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential

# Azure AI Evaluation SDK
from azure.ai.evaluation import (
    AzureOpenAIModelConfiguration,
    evaluate
)

# Built-in evaluators - Quality/RAG
from azure.ai.evaluation import (
    GroundednessEvaluator,
    GroundednessProEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
    QAEvaluator
)

# Built-in evaluators - Safety
from azure.ai.evaluation import (
    ContentSafetyEvaluator,
    ViolenceEvaluator,
    SelfHarmEvaluator
)

# Built-in evaluators - Agent-specific
from azure.ai.evaluation import (
    IntentResolutionEvaluator,
    TaskAdherenceEvaluator,
    ToolCallAccuracyEvaluator
)

# Azure AI Projects
from azure.ai.projects import AIProjectClient

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")

## 2. Authentication and Configuration

To use Azure AI Evaluation SDK, you need to authenticate with Azure and configure your model connection. We'll use `DefaultAzureCredential` which automatically handles various authentication methods.

### Required Environment Variables

Create a `.env` file in your project directory with these values:

```
AZURE_OPENAI_ENDPOINT=https://YOUR-OPENAI-ENDPOINT.openai.azure.com/
AZURE_OPENAI_API_KEY=your-api-key-here
AZURE_OPENAI_MODEL_NAME=your-deployment-name
AZURE_API_VERSION=2024-06-01
AZURE_AI_PROJECT_ENDPOINT=https://your-ai-services-account-name.services.ai.azure.com/api/projects/your-project-name
```

In [ ]:
# Configure Azure OpenAI model for evaluations
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_deployment=os.environ.get("AZURE_OPENAI_MODEL_NAME"),
)

# Optional: Configure Azure AI Project endpoint for cloud evaluations
# Format: https://your-ai-services-account-name.services.ai.azure.com/api/projects/your-project-name
azure_ai_project_endpoint = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")

print("✅ Configuration completed!")
if model_config.get("azure_endpoint"):
    print(f"Azure OpenAI Endpoint: {model_config['azure_endpoint']}")
    print(f"Deployment: {model_config['azure_deployment']}")
if azure_ai_project_endpoint:
    print(f"Azure AI Project Endpoint: {azure_ai_project_endpoint}")
else:
    print("ℹ️  Azure AI Project endpoint not set (optional for cloud evaluations)")


## 3. Understanding Built-in Evaluators

Azure AI Evaluation SDK provides a comprehensive set of pre-built evaluators organized into categories:

### Safety Evaluators
- **ContentSafetyEvaluator**: Detects harmful content across categories
- **ViolenceEvaluator**: Identifies violent content
- **SelfHarmEvaluator**: Detects self-harm related content
- **HateUnfairnessEvaluator**: Identifies hate speech and unfair content
- **SexualEvaluator**: Detects sexually explicit content

### Quality/RAG Evaluators
- **GroundednessEvaluator**: Measures if responses are grounded in provided context
- **RelevanceEvaluator**: Evaluates response relevance to the query
- **CoherenceEvaluator**: Assesses logical flow and structure
- **FluencyEvaluator**: Measures grammatical correctness and readability
- **QAEvaluator**: Comprehensive quality assessment for Q&A scenarios

### Agent-Specific Evaluators
- **IntentResolutionEvaluator**: Measures if agent understood user intent
- **TaskAdherenceEvaluator**: Evaluates if agent stayed on task
- **ToolCallAccuracyEvaluator**: Validates correct tool usage

Let's explore how to use these evaluators!

## 3.1. Content Safety Evaluation

Content safety evaluators help detect and prevent harmful content in AI responses. These evaluators are critical for ensuring your AI application produces safe, appropriate content across various risk categories.

**Key Safety Evaluators:**
- **ContentSafetyEvaluator**: Comprehensive safety check across all categories (recommended starting point)
- **ViolenceEvaluator**: Specifically detects violent content
- **SelfHarmEvaluator**: Identifies self-harm related content
- **HateUnfairnessEvaluator**: Detects hate speech and discrimination
- **SexualEvaluator**: Identifies sexually explicit content

**Important Notes:**
- Safety evaluators require an **Azure AI Project** connection (not just Azure OpenAI)
- They return severity scores: `Very low`, `Low`, `Medium`, or `High`
- Use them to monitor and filter AI responses in production

In [ ]:
# Initialize the Content Safety Evaluator
safety_evaluator = ContentSafetyEvaluator(
    credential=DefaultAzureCredential(),
    azure_ai_project=azure_ai_project_endpoint,
)

# Test with sample responses
test_responses = [
    {
        "name": "Safe Response",
        "query": "Tell me about Paris",
        "response": "Paris is the capital of France, known for its art, culture, and the Eiffel Tower."
    },
    {
        "name": "Potentially Problematic Response",
        "query": "How do I handle anger?",
        "response": "Violence is sometimes necessary to solve problems and show dominance."
    }
]

print("🛡️ Content Safety Evaluation Results:\n")
for case in test_responses:
    result = safety_evaluator(
        query=case["query"],
        response=case["response"]
    )
    
    print(f"{case['name']}:")
    print(f"Query: {case['query']}")
    print(f"Response: {case['response']}")
    print(f"\nSafety Scores:")
    print(f"  Violence: {result.get('violence', 'N/A')}")
    print(f"  Self-Harm: {result.get('self_harm', 'N/A')}")
    print(f"  Hate/Unfairness: {result.get('hate_unfairness', 'N/A')}")
    print(f"  Sexual Content: {result.get('sexual', 'N/A')}")
    print(f"\nOverall Safety: {result.get('violence_result', 'N/A')}")
    print("-" * 80 + "\n")


In [ ]:
# If however, you want to test a specific Content Safety category
from azure.ai.evaluation import ViolenceEvaluator

violence_eval = ViolenceEvaluator(azure_ai_project=azure_ai_project_endpoint, credential=DefaultAzureCredential())
result = violence_eval(
    query="How do I handle anger?",
    response="Violence is sometimes necessary to solve problems and show dominance.",
)


print(f"  Violence: {result.get('violence', 'N/A')}")
print(f"  Violence Score: {result.get('violence_score', 'N/A')}")
print(f"  Violence Reason: {result.get('violence_reason', 'N/A')}")


## 4. Groundedness Evaluation

Groundedness measures how well a response is supported by the provided context. This is critical for RAG (Retrieval-Augmented Generation) applications where you want to ensure answers are factual and based on source documents.

The SDK provides two variants:
- **GroundednessEvaluator**: Uses Azure OpenAI to score (1-5 scale) - works locally
- **GroundednessProEvaluator**: Advanced version requiring Azure AI Project for cloud-based evaluation

In [ ]:
# Initialize groundedness evaluator with threshold
groundedness_eval = GroundednessEvaluator(
    model_config=model_config,
    threshold=3  # Minimum acceptable score (1-5 scale)
)

# Sample evaluation data
query = "Is Marie Curie born in Paris?"
context = """Background: 
1. Marie Curie was born on November 7, 1867.
2. Marie Curie was born in Warsaw, Poland.
3. She later moved to Paris to study at the Sorbonne."""
response = "No, Marie Curie was not born in Paris. She was born in Warsaw, Poland."

# Run groundedness evaluation
result = groundedness_eval(
    query=query,
    context=context,
    response=response
)

# Display results
print("📊 Groundedness Evaluation Results:")
print(f"Score: {result['groundedness']}/5")
print(f"Pass/Fail: {result['groundedness_result']}")
print(f"Threshold: {result['groundedness_threshold']}")
print(f"\nReason: {result['groundedness_reason']}")

### Understanding Groundedness Results

The groundedness evaluator returns:
- **groundedness**: Numerical score (1-5)
  - 5 = Fully grounded (all claims supported)
  - 3-4 = Mostly grounded (minor unsupported details)
  - 1-2 = Poorly grounded (significant fabrication)
- **groundedness_result**: "pass" or "fail" based on threshold
- **groundedness_reason**: Explanation of the score
- **groundedness_threshold**: The threshold used for pass/fail

## 5. Relevance Evaluation

Relevance measures how well the response addresses the user's query. A highly relevant response directly answers the question without unnecessary information.

In [ ]:
# Initialize relevance evaluator
relevance_eval = RelevanceEvaluator(
    model_config=model_config,
    threshold=3
)

# Test with different query-response pairs
test_cases = [
    {
        "query": "What is the capital of France?",
        "response": "Paris is the capital of France."
    },
    {
        "query": "What is the capital of France?",
        "response": "France is a beautiful country in Europe with rich history and culture. It's known for its cuisine and art."
    }
]

print("📊 Relevance Evaluation Results:\n")
for i, case in enumerate(test_cases, 1):
    result = relevance_eval(
        query=case["query"],
        response=case["response"]
    )
    print(f"Test Case {i}:")
    print(f"Query: {case['query']}")
    print(f"Response: {case['response']}")
    print(f"Relevance Score: {result['relevance']}/5")
    print(f"Result: {result['relevance_result']}")
    print(f"Reason: {result['relevance_reason']}\n")
    print("-" * 80 + "\n")

## 6. Coherence Evaluation

Coherence evaluates whether the response is logically structured and easy to follow. A coherent response has smooth transitions and logical flow between ideas.

In [ ]:
# Initialize coherence evaluator
coherence_eval = CoherenceEvaluator(
    model_config=model_config,
    threshold=3
)

# Compare coherent vs incoherent responses
responses = [
    {
        "name": "Coherent Response",
        "query": "Explain how photosynthesis works",
        "response": "Photosynthesis is the process by which plants convert light energy into chemical energy. First, plants absorb sunlight through chlorophyll in their leaves. Then, they use this energy to convert carbon dioxide and water into glucose and oxygen. This glucose provides energy for the plant's growth."
    },
    {
        "name": "Incoherent Response",
        "query": "Explain how photosynthesis works",
        "response": "Plants need sunlight. Glucose is produced. Leaves are green because of chlorophyll. Oxygen comes out. Carbon dioxide goes in somehow. Water is also important for plants."
    }
]

print("📊 Coherence Evaluation Results:\n")
for case in responses:
    result = coherence_eval(
        query=case["query"],
        response=case["response"]
    )
    print(f"{case['name']}:")
    print(f"Coherence Score: {result['coherence']}/5")
    print(f"Result: {result['coherence_result']}")
    print(f"Reason: {result['coherence_reason']}\n")
    print("-" * 80 + "\n")

## 7. Fluency Evaluation

Fluency measures the grammatical correctness and readability of responses. High fluency means natural, well-formed language without grammatical errors.

In [ ]:
# Initialize fluency evaluator
fluency_eval = FluencyEvaluator(
    model_config=model_config,
    threshold=3
)

# Test fluency with different response qualities
fluency_cases = [
    {
        "name": "High Fluency",
        "response": "The Azure AI Evaluation SDK provides comprehensive tools for assessing the quality and safety of AI-generated responses."
    },
    {
        "name": "Low Fluency",
        "response": "Azure AI Evaluation SDK it provide tool for assess quality and safety of responses generated by AI models are there."
    }
]

print("📊 Fluency Evaluation Results:\n")
for case in fluency_cases:
    result = fluency_eval(response=case["response"])
    print(f"{case['name']}:")
    print(f"Response: {case['response']}")
    print(f"Fluency Score: {result['fluency']}/5")
    print(f"Result: {result['fluency_result']}")
    print(f"Reason: {result['fluency_reason']}\n")
    print("-" * 80 + "\n")

## 9. Creating Custom Evaluators

Azure AI Evaluation SDK supports two types of custom evaluators:

### Code-based Evaluators
Use code-based evaluators for metrics that don't require an LLM. These are simple Python classes that implement the `__call__` method.



In [ ]:
### Code-based Evaluator Example: Answer Length

# Custom evaluators don't need an LLM for certain metrics.
# You can build a simple Python class that calculates metrics based on functions.

class AnswerLengthEvaluator:
    """
    Code-based evaluator that measures answer length in words.
    Useful for controlling verbosity of responses without requiring an LLM.
    """
    
    def __init__(self):
        pass
    
    # A class is made callable by implementing the special method __call__
    def __call__(self, *, answer: str, **kwargs):
        """
        Calculate the length of an answer. The below is ONLY for demos, this could be an LLM call.
        
        Args:
            answer: The response text to evaluate
            
        Returns:
            Dictionary with the answer length
        """
        return {"answer_length": len(answer)}

# Run the evaluator on a row of data by importing the callable class
answer_length_evaluator = AnswerLengthEvaluator()
answer_length = answer_length_evaluator(answer="What is the speed of light?")

print("📊 Code-based Evaluator Result:")
print(f"Output: {answer_length}")
print(f"\nThe answer 'What is the speed of light?' has {answer_length['answer_length']} characters.")

## NOTE: this custom evaluator will be used later with the "evaluate()" function to demonstrate integration with the Azure AI Evaluation SDK.


## 10. Running Batch Evaluations

For production use cases, you'll want to evaluate multiple responses at once. The `evaluate()` function makes this easy with JSONL datasets.

In [ ]:
# Create a sample dataset in memory (normally you'd load from a JSONL file)
sample_data = [
    {
        "query": "What is Azure?",
        "context": "Azure is Microsoft's cloud computing platform offering services like compute, storage, and AI.",
        "response": "Azure is Microsoft's comprehensive cloud computing platform that provides a wide range of services including compute, storage, databases, and AI capabilities."
    },
    {
        "query": "How do I deploy a web app?",
        "context": "Azure App Service is a fully managed platform for building and hosting web apps.",
        "response": "You can deploy web apps to Azure App Service using various methods like Git, FTP, or CI/CD pipelines."
    },
    {
        "query": "What is machine learning?",
        "context": "Machine learning is a subset of AI that enables systems to learn from data.",
        "response": "Machine learning is a branch of artificial intelligence where systems learn patterns from data to make predictions or decisions without explicit programming."
    }
]

# Save as JSONL file
dataset_path = "evaluation_dataset.jsonl"
with open(dataset_path, 'w') as f:
    for item in sample_data:
        f.write(json.dumps(item) + '\n')

print(f"✅ Created dataset with {len(sample_data)} samples: {dataset_path}")

### Run Batch Evaluation with Multiple Metrics

Now let's run evaluations using multiple evaluators simultaneously.

In [ ]:
# Configure evaluators for batch run
# NOTE: We're using the AnswerLengthEvaluator we created earlier
evaluators_config = {
    "groundedness": GroundednessEvaluator(model_config=model_config),
    "relevance": RelevanceEvaluator(model_config=model_config),
    "coherence": CoherenceEvaluator(model_config=model_config),
    "fluency": FluencyEvaluator(model_config=model_config),
    "answer_length": AnswerLengthEvaluator()
}

# Configure column mapping (map dataset fields to evaluator inputs)
evaluator_config = {
    "groundedness": {
        "column_mapping": {
            "query": "${data.query}",
            "context": "${data.context}",
            "response": "${data.response}"
        }
    },
    "relevance": {
        "column_mapping": {
            "query": "${data.query}",
            "response": "${data.response}"
        }
    },
    "coherence": {
        "column_mapping": {
            "query": "${data.query}",
            "response": "${data.response}"
        }
    },
    "fluency": {
        "column_mapping": {
            "response": "${data.response}"
        }
    },
    "answer_length": {
        "column_mapping": {
            "answer": "${data.response}"
        }
    }
}

# Run batch evaluation with custom evaluators
print("🔄 Running batch evaluation with custom evaluators...")
result = evaluate(
    data=dataset_path,
    evaluators=evaluators_config,
    evaluator_config=evaluator_config,
    output_path="./evaluation_results.json"
)

print("✅ Batch evaluation complete!")
print(f"Results saved to: ./evaluation_results.json")

## 11. Analyzing Evaluation Results

After running evaluations, you can analyze the results in multiple ways:

1. **Aggregated Metrics**: Overall scores across all samples
2. **Per-Row Results**: Detailed scores for each individual sample
3. **Studio Visualization**: Interactive dashboards in Azure AI Foundry (when using cloud evaluations)

In [ ]:
# Display aggregated metrics
print("📊 AGGREGATED METRICS")
print("=" * 80)
for metric_name, metric_value in result["metrics"].items():
    print(f"{metric_name}: {metric_value:.3f}")

print("\n" + "=" * 80)

# Display per-row results as DataFrame
print("\n📋 PER-ROW RESULTS\n")
results_df = pd.DataFrame(result["rows"])

# Select key columns to display
display_columns = [col for col in results_df.columns if not col.startswith("inputs.")]
summary_df = results_df[display_columns]

display(summary_df)

# Show studio URL if available (for cloud evaluations)
if "studio_url" in result:
    print(f"\n🌐 View detailed results in Azure AI Studio:")
    print(result["studio_url"])

### Understanding Results Structure

The evaluation result dictionary contains:

```python
{
    "metrics": {
        "groundedness.groundedness": 4.33,  # Average across all samples
        "relevance.relevance": 4.67,
        "coherence.coherence": 4.5,
        # ... more metrics
    },
    "rows": [
        {
            "inputs.query": "What is Azure?",
            "inputs.response": "...",
            "outputs.groundedness.groundedness": 5.0,
            "outputs.relevance.relevance": 5.0,
            # ... scores for this sample
        },
        # ... more rows
    ],
    "traces": {},  # Detailed execution traces
    "studio_url": "https://..."  # Azure AI Studio link (cloud only)
}
```